# VisualNews Train Embeddings

Production code is prepared for manual Colab execution. The completed 100-image/text smoke test is recorded in `outputs/large_corpus/visualnews_train_smoke_test_summary.json`; do not repeat it.

## Setup

In [27]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [28]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Paths

In [29]:
from pathlib import Path

drive_root = Path(
    "/content/drive/MyDrive/MajorProjectAssets"
)

print("Drive root exists:", drive_root.exists())
print("Drive root:", drive_root)

Drive root exists: True
Drive root: /content/drive/MyDrive/MajorProjectAssets


In [30]:
for path in sorted(drive_root.iterdir()):
    print(path.name)

archive
embeddings
manifests
outputs
pilot_images


In [31]:
pilot_embedding_dir = drive_root / "embeddings" / "pilot"
manifest_dir = drive_root / "manifests"
pilot_output_dir = drive_root / "outputs" / "pilot" / "faiss_retrieval"

print("Pilot embeddings:", pilot_embedding_dir.exists())
print("Manifests:", manifest_dir.exists())
print("Pilot retrieval:", pilot_output_dir.exists())

Pilot embeddings: True
Manifests: True
Pilot retrieval: True


In [32]:
archive_dir = drive_root / "archive"

for path in sorted(archive_dir.iterdir()):
    print(path.name)

In [33]:
import shutil

total, used, free = shutil.disk_usage("/content")

print("Total GB:", round(total / 1024**3, 2))
print("Used GB:", round(used / 1024**3, 2))
print("Free GB:", round(free / 1024**3, 2))

Total GB: 118.86
Used GB: 48.18
Free GB: 70.66


In [34]:
large_embedding_dir = (
    drive_root
    / "embeddings"
    / "visualnews_train"
)

large_output_dir = (
    drive_root
    / "outputs"
    / "large_corpus"
)

image_chunk_dir = (
    large_embedding_dir
    / "image_chunks"
)

text_chunk_dir = (
    large_embedding_dir
    / "text_chunks"
)

large_retrieval_dir = (
    large_output_dir
    / "faiss_retrieval"
)

image_chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)

text_chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)

large_retrieval_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [35]:
GPU_BATCH_SIZE = 64
CHECKPOINT_SIZE = 5000

# Enable only the stage you intend to run manually.
RUN_IMAGE_PRODUCTION = False
RUN_TEXT_PRODUCTION = False
RUN_MERGE_IMAGES = False
RUN_MERGE_TEXT = False
RUN_FINAL_VALIDATION = False

image_progress_path = large_embedding_dir / "image_embedding_progress.json"
text_progress_path = large_embedding_dir / "text_embedding_progress.json"
image_failures_path = large_embedding_dir / "image_embedding_failures.csv"
image_final_path = large_embedding_dir / "evidence_image_embeddings.npy"
text_final_path = large_embedding_dir / "evidence_text_embeddings.npy"
evidence_map_path = large_embedding_dir / "evidence_embedding_map.parquet"

## Helpers

Reuse the tested checkpoint writer and archive/path checks. The image loop, batch encoding, text loop and merge code are shown below.

Copy the prepared local `outputs/large_corpus/visualnews_train_production_runtime.zip` to the same Drive directory before running Colab. It contains only three Python scripts; it does not replace the smoke runtime bundle.

In [ ]:
import hashlib
import json
import sys
import tempfile
import warnings
import zipfile
from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

runtime_bundle = large_output_dir / "visualnews_train_production_runtime.zip"
expected_bundle_sha256 = "148ac78cc11c7f8581917b8c1eab928de3940dfba113bce78cee04a3d525b347"
if hashlib.sha256(runtime_bundle.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError("STOP: production runtime ZIP differs from the reviewed code")

runtime_code_dir = Path("/content/visualnews_train_production_runtime")
runtime_code_dir.mkdir(exist_ok=True)
code_names = ['visualnews_stream_embeddings.py', 'extract_selected_images.py', 'check_missing_assets.py']
with zipfile.ZipFile(runtime_bundle) as bundle:
    if sorted(bundle.namelist()) != sorted(code_names):
        raise ValueError("STOP: unexpected runtime ZIP contents")
    for name in code_names:
        (runtime_code_dir / name).write_bytes(bundle.read(name))
sys.path.insert(0, str(runtime_code_dir))

# Reload updated helpers if this kernel was used for the earlier smoke test.
for name in code_names:
    sys.modules.pop(Path(name).stem, None)
from visualnews_stream_embeddings import (
    EmbeddingCheckpoints, encoder_signature, file_sha256, load_compatible_encoder,
    mapping_record, read_validated_chunk, validate_merged_embeddings,
    validate_train_manifest, write_json_atomic,
)
from extract_selected_images import canonical_member_name, open_archive
from check_missing_assets import validate_image_path

## Manifest

In [ ]:
train_manifest_path = manifest_dir / "visualnews_train_evidence.parquet"
manifest_sha256 = file_sha256(train_manifest_path)
if manifest_sha256 != "6bd26d4371c69c16a92c512b2d5862579fbe26ff52775a74dda5f5831eb3d1fa":
    raise ValueError("STOP: TRAIN manifest hash differs from the validated manifest")

train_manifest = pd.read_parquet(train_manifest_path)
validate_train_manifest(train_manifest)
assert len(train_manifest) == 385003
assert train_manifest.id.is_unique and train_manifest.image_path.is_unique
assert train_manifest.split.eq("train").all()
print("TRAIN records:", len(train_manifest))

## Model

In [ ]:
pilot_config_path = pilot_embedding_dir / "clip_embedding_config.json"
pilot_config = json.loads(pilot_config_path.read_text())
model_config = encoder_signature(pilot_config)

expected_pilot_values = {
    "model_name": "ViT-B-32", "pretrained": "openai", "force_quick_gelu": True,
    "activation": "QuickGELU", "embedding_dimension": 512,
    "saved_dtype": "float32", "frozen": True,
}
for name, expected in expected_pilot_values.items():
    if pilot_config.get(name) != expected:
        raise ValueError("STOP: incompatible frozen pilot configuration: " + name)
print("Pilot encoder:", pilot_config["model_name"], pilot_config["pretrained"])

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess

try:
    version("open_clip_torch")
except PackageNotFoundError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "open_clip_torch==" + pilot_config["openclip_version"],
    ])

# Checks exact version/preprocessing/tokenizer, T4, fp32, QuickGELU and 512-D output.
model, image_preprocess, text_tokenizer = load_compatible_encoder(pilot_config)
model.eval()
model.requires_grad_(False)
assert repr(image_preprocess) == pilot_config["image_preprocessing"]
print("T4 encoder loaded; frozen preprocessing matches the pilot.")

In [ ]:
frozen_pilot_files = list(pilot_embedding_dir.rglob("*"))
frozen_pilot_files += list(pilot_output_dir.rglob("*"))
frozen_pilot_files += [path for path in manifest_dir.iterdir()
                       if path.is_file() and path != train_manifest_path]
frozen_pilot_hashes = {}
for path in frozen_pilot_files:
    if path.is_file():
        frozen_pilot_hashes[str(path)] = file_sha256(path)
print("Protected pilot files:", len(frozen_pilot_hashes))

In [ ]:
def normalized_vectors(features):
    features = features.float()
    norms = features.norm(dim=1, keepdim=True)
    if not torch.isfinite(features).all().item() or not torch.isfinite(norms).all().item():
        raise RuntimeError("STOP: nonfinite model output")
    if not (norms > 0).all().item():
        raise RuntimeError("STOP: zero-length model output")
    return (features / norms).cpu().numpy().astype(np.float32, copy=False)


def check_frozen_model():
    if model.training or any(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("STOP: model must remain frozen and in evaluation mode")

## Image Progress

Production chunks are directly in `image_chunks/`; existing `smoke_test/` files are excluded. The writer validates chunk hashes/maps and rebuilds completed ID/path sets. Its JSON contains counts, settings and a manifest/model scope check.

In [ ]:
if RUN_IMAGE_PRODUCTION:
    image_checkpoints = EmbeddingCheckpoints(
        image_chunk_dir, image_progress_path, train_manifest,
        manifest_sha256, model_config, "image", CHECKPOINT_SIZE,
        failure_log_format="csv", gpu_batch_size=GPU_BATCH_SIZE,
    )
    print("Completed images:", len(image_checkpoints.completed_paths))
    print("Failures CSV:", image_checkpoints.failures_path)
else:
    print("Image production disabled.")

In [ ]:
if RUN_IMAGE_PRODUCTION:
    image_targets = {}
    image_fields = ["id", "image_path", "source", "split"]
    for row in train_manifest[image_fields].to_dict("records"):
        if row["image_path"] not in image_checkpoints.completed_paths:
            archive_name = row["image_path"].removeprefix("visual_news/")
            image_targets[archive_name] = row
    matched_paths = set()
    image_tensors = []
    image_rows = []
    image_run = {"found": 0, "encoded": 0, "failed": 0, "scanned": 0, "consecutive_failures": 0}
    print("Unfinished image targets:", len(image_targets))

## Stream Images

One remote `origin.tar` stream per successful run. Decode only selected regular files into RGB in memory. GPU errors stop; image decode failures are logged. Stop at five consecutive failures or more than 10% failures after 20 targets.

In [ ]:
def read_image_tensor(archive, member):
    maximum_bytes = 32 * 1024 * 1024
    if not member.isfile() or member.size <= 0 or member.size > maximum_bytes:
        raise ValueError("Image must be a bounded, nonempty regular file")
    source = archive.extractfile(member)
    if source is None:
        raise ValueError("Image member cannot be read")
    with source:
        image_bytes = source.read(maximum_bytes + 1)
    if len(image_bytes) != member.size:
        raise ValueError("Incomplete image bytes")
    with warnings.catch_warnings():
        warnings.simplefilter("error", Image.DecompressionBombWarning)
        with Image.open(BytesIO(image_bytes)) as image:
            with image.convert("RGB") as rgb_image:
                tensor = image_preprocess(rgb_image)
    return tensor

In [ ]:
def encode_image_tensors(tensors):
    check_frozen_model()
    with torch.inference_mode():
        gpu_images = torch.stack(tensors).to("cuda")
        features = model.encode_image(gpu_images)
        embeddings = normalized_vectors(features)
    del gpu_images, features
    return embeddings


def save_image_batch():
    if not image_tensors:
        return
    embeddings = encode_image_tensors(image_tensors)
    image_checkpoints.add(image_rows, embeddings)
    image_run["encoded"] += len(image_rows)
    image_tensors.clear()
    image_rows.clear()

In [ ]:
def handle_image_member(archive, member, archive_name):
    if archive_name in matched_paths:
        raise ValueError("STOP: duplicate target in archive: " + archive_name)
    matched_paths.add(archive_name)
    row = image_targets[archive_name]
    image_run["found"] += int(member.isfile())
    try:
        tensor = read_image_tensor(archive, member)
    except (OSError, ValueError, SyntaxError, EOFError, UnidentifiedImageError,
            Image.DecompressionBombError, Image.DecompressionBombWarning) as error:
        image_checkpoints.record_failure(row, error)
        image_run["failed"] += 1
        image_run["consecutive_failures"] += 1
        failure_fraction = image_run["failed"] / len(matched_paths)
        too_many = len(matched_paths) >= 20 and failure_fraction > 0.10
        if image_run["consecutive_failures"] >= 5 or too_many:
            raise RuntimeError("STOP: too many image failures; inspect failures CSV") from error
    else:
        image_run["consecutive_failures"] = 0
        image_tensors.append(tensor)
        image_rows.append(mapping_record(row))
        if len(image_tensors) == GPU_BATCH_SIZE:
            save_image_batch()

In [ ]:
def check_member_layout(member, archive_name, recognized):
    if recognized:
        return True
    if member.isfile():
        try:
            validate_image_path("visual_news/" + archive_name)
            return True
        except ValueError:
            pass
    if image_run["scanned"] >= 100000:
        raise RuntimeError("STOP: archive path layout unrecognized")
    return False

In [ ]:
def finish_image_stream():
    save_image_batch()
    image_checkpoints.flush()
    for name in sorted(set(image_targets) - matched_paths):
        row = image_targets[name]
        image_checkpoints.record_failure(row, FileNotFoundError("Target absent from archive"))
        image_run["failed"] += 1
    complete = len(image_checkpoints.completed_paths) == len(train_manifest)
    image_checkpoints.save_progress("complete" if complete else "incomplete", last_run=image_run)
    if not complete:
        raise RuntimeError("STOP: unfinished images remain; inspect CSV and resume before merging")

In [ ]:
ARCHIVE_URL = "https://www.cs.rice.edu/~vo9/visualnews/origin.tar"

if RUN_IMAGE_PRODUCTION and image_targets:
    recognized_layout = False
    image_checkpoints.state["archive_passes_opened"] += 1
    image_checkpoints.save_progress("streaming")
    try:
        with open_archive(archive_url=ARCHIVE_URL) as (archive, remote_size):
            for member in archive:
                archive.members.clear()  # Avoid retaining every TarInfo in Python 3.12.
                image_run["scanned"] += 1
                image_checkpoints.state["scanned_archive_member_count"] += 1
                try:
                    archive_name = canonical_member_name(member.name)
                except ValueError:
                    continue
                recognized_layout = check_member_layout(member, archive_name, recognized_layout)
                if archive_name in image_targets:
                    handle_image_member(archive, member, archive_name)
                if image_run["scanned"] % 10000 == 0:
                    image_checkpoints.save_progress("streaming", last_run=image_run)
                    print("Scanned:", image_run["scanned"], "Encoded:", image_run["encoded"])
                if len(matched_paths) == len(image_targets):
                    break
        finish_image_stream()
    except BaseException:
        image_checkpoints.flush()  # Preserve already encoded vectors, never retry a GPU batch here.
        image_checkpoints.save_progress("interrupted", last_run=image_run)
        raise
    finally:
        image_tensors.clear()
        image_rows.clear()
elif RUN_IMAGE_PRODUCTION:
    image_checkpoints.save_progress("complete")
    print("All images already complete; no archive opened.")
else:
    print("Remote image stream not started.")

## Text Embeddings

Captions use the same frozen encoder. This separate stage never opens the archive and skips completed caption IDs/paths.

In [ ]:
def encode_caption_batch(captions):
    check_frozen_model()
    with torch.inference_mode():
        tokens = text_tokenizer(captions).to("cuda")
        features = model.encode_text(tokens)
        embeddings = normalized_vectors(features)
    del tokens, features
    return embeddings

In [ ]:
if RUN_TEXT_PRODUCTION:
    text_checkpoints = EmbeddingCheckpoints(
        text_chunk_dir, text_progress_path, train_manifest,
        manifest_sha256, model_config, "text", CHECKPOINT_SIZE,
        failure_log_format="csv", gpu_batch_size=GPU_BATCH_SIZE,
    )
    remaining_captions = train_manifest.loc[
        ~train_manifest.image_path.isin(text_checkpoints.completed_paths)
    ]
    print("Unfinished captions:", len(remaining_captions))
else:
    print("Text production disabled.")

In [ ]:
if RUN_TEXT_PRODUCTION:
    try:
        for start in range(0, len(remaining_captions), GPU_BATCH_SIZE):
            caption_batch = remaining_captions.iloc[start:start + GPU_BATCH_SIZE]
            embeddings = encode_caption_batch(caption_batch.caption.tolist())
            rows = []
            for row in caption_batch[["id", "image_path", "source", "split"]].to_dict("records"):
                rows.append(mapping_record(row))
            text_checkpoints.add(rows, embeddings)
        text_checkpoints.flush()
        text_checkpoints.save_progress("complete")
        print("Completed captions:", len(text_checkpoints.completed_ids))
    except BaseException:
        text_checkpoints.flush()
        text_checkpoints.save_progress("interrupted")
        raise
else:
    print("Full caption encoding not started.")

## Merge Image Chunks

Both modalities use original metadata ID ascending as final row order. The common map retains the manifest fields. Only one checkpoint-sized embedding array is read at a time; the final array is written with NumPy memmap.

Merge requires all 385,003 targets. Missing/corrupt/duplicate chunks stop. Existing final arrays are never overwritten; interrupted temporary merge files are discarded.

In [ ]:
ordered_manifest = train_manifest.sort_values("id").reset_index(drop=True)
evidence_map = ordered_manifest.rename(columns={"id": "metadata_id"})
evidence_map.insert(0, "embedding_row", range(len(evidence_map)))
id_to_final_row = dict(zip(evidence_map.metadata_id, evidence_map.embedding_row))
target_lookup = train_manifest.set_index("id")[["image_path", "source", "split"]]
selection_csv = train_manifest[["id", "image_path"]].sort_values("id").to_csv(index=False)
target_selection_sha256 = hashlib.sha256(selection_csv.encode()).hexdigest()


def expected_chunk_signature(modality):
    return {
        "manifest_sha256": manifest_sha256, "model_config": model_config,
        "modality": modality, "target_count": len(train_manifest),
        "target_selection_sha256": target_selection_sha256,
    }

In [ ]:
def validated_chunks(modality):
    chunk_dir = image_chunk_dir if modality == "image" else text_chunk_dir
    arrays = sorted(chunk_dir.glob(modality + "_embeddings_chunk_*.npy"))
    if not arrays:
        raise ValueError("STOP: no completed production " + modality + " chunks")
    expected_files = set()
    for number, array_path in enumerate(arrays):
        suffix = f"chunk_{number:05d}"
        if array_path.name != f"{modality}_embeddings_{suffix}.npy":
            raise ValueError("STOP: chunk numbering has a gap")
        map_path = chunk_dir / f"{modality}_embedding_map_{suffix}.parquet"
        marker_path = chunk_dir / f"{modality}_{suffix}.complete.json"
        expected_files.update([array_path.name, map_path.name, marker_path.name])
    actual_files = set()
    for path in chunk_dir.glob(modality + "_*chunk_*"):
        if path.suffix != ".tmp":
            actual_files.add(path.name)
    if actual_files != expected_files:
        raise ValueError("STOP: incomplete or unexpected production chunk files")
    for number, array_path in enumerate(arrays):
        suffix = f"chunk_{number:05d}"
        map_path = chunk_dir / f"{modality}_embedding_map_{suffix}.parquet"
        marker_path = chunk_dir / f"{modality}_{suffix}.complete.json"
        yield read_validated_chunk(array_path, map_path, marker_path, target_lookup,
                                   expected_chunk_signature(modality))

In [ ]:
def merge_chunks(modality, final_path):
    marker_path = final_path.with_suffix(".complete.json")
    if final_path.exists() or marker_path.exists():
        raise FileExistsError("Final artifact exists; use Final Validation instead of overwriting it")
    with tempfile.NamedTemporaryFile(dir=final_path.parent, suffix=".npy", delete=False) as handle:
        temporary_path = Path(handle.name)
    seen_ids = set()
    seen_paths = set()
    try:
        merged = np.lib.format.open_memmap(
            temporary_path, mode="w+", dtype=np.float32, shape=(len(evidence_map), 512),
        )
        for embeddings, chunk_map in validated_chunks(modality):
            chunk_ids = set(chunk_map.metadata_id)
            chunk_paths = set(chunk_map.image_path)
            if seen_ids.intersection(chunk_ids) or seen_paths.intersection(chunk_paths):
                raise ValueError("STOP: duplicate targets across chunks")
            final_rows = []
            for metadata_id in chunk_map.metadata_id:
                final_rows.append(id_to_final_row[metadata_id])
            merged[final_rows] = embeddings
            seen_ids.update(chunk_ids)
            seen_paths.update(chunk_paths)
        if seen_ids != set(evidence_map.metadata_id) or seen_paths != set(evidence_map.image_path):
            raise ValueError("STOP: missing production targets; finish embeddings before merging")
        merged.flush()
        del merged
        validation = validate_merged_embeddings(temporary_path, len(evidence_map))
        if final_path.exists() or marker_path.exists():
            raise FileExistsError("Final artifact appeared during merge; use one writer")
        temporary_path.replace(final_path)
        write_json_atomic(marker_path, {
            "signature": expected_chunk_signature(modality), "ordering": "metadata_id ascending",
            "embedding_sha256": file_sha256(final_path), "validation": validation,
        })
        print("Merged:", final_path.name, validation)
    finally:
        temporary_path.unlink(missing_ok=True)

In [ ]:
if RUN_MERGE_IMAGES:
    merge_chunks("image", image_final_path)
else:
    print("Image consolidation not started.")

## Merge Text Chunks

In [ ]:
if RUN_MERGE_TEXT:
    merge_chunks("text", text_final_path)
else:
    print("Text consolidation not started.")

## Final Validation

Run only after both merges finish. Check arrays in blocks, verify their completion hashes and shared row order, save/reload the common map, and confirm pilot hashes are unchanged.

In [ ]:
if RUN_FINAL_VALIDATION:
    final_validations = {}
    for modality, path in [("image", image_final_path), ("text", text_final_path)]:
        marker_path = path.with_suffix(".complete.json")
        if not path.is_file() or not marker_path.is_file():
            raise ValueError("STOP: both production arrays and completion markers are required")
        marker = json.loads(marker_path.read_text())
        if marker["signature"] != expected_chunk_signature(modality):
            raise ValueError("STOP: final manifest/model scope mismatch")
        if marker["ordering"] != "metadata_id ascending":
            raise ValueError("STOP: final row ordering mismatch")
        if file_sha256(path) != marker["embedding_sha256"]:
            raise ValueError("STOP: final embedding checksum mismatch")
        validation = validate_merged_embeddings(path, len(train_manifest))
        if validation != marker["validation"]:
            raise ValueError("STOP: final validation differs from completion marker")
        final_validations[modality] = validation
else:
    print("Final artifact validation not started.")

In [ ]:
if RUN_FINAL_VALIDATION:
    assert evidence_map.metadata_id.is_unique and evidence_map.image_path.is_unique
    assert evidence_map.embedding_row.tolist() == list(range(len(train_manifest)))
    assert evidence_map.metadata_id.tolist() == sorted(train_manifest.id.tolist())
    if evidence_map_path.exists():
        pd.testing.assert_frame_equal(evidence_map, pd.read_parquet(evidence_map_path))
    else:
        with tempfile.NamedTemporaryFile(dir=large_embedding_dir, suffix=".parquet", delete=False) as handle:
            temporary_map_path = Path(handle.name)
        try:
            evidence_map.to_parquet(temporary_map_path, index=False, compression="zstd")
            pd.testing.assert_frame_equal(evidence_map, pd.read_parquet(temporary_map_path))
            if evidence_map_path.exists():
                raise FileExistsError("Mapping appeared during save; use one writer")
            temporary_map_path.replace(evidence_map_path)
        finally:
            temporary_map_path.unlink(missing_ok=True)
    print("Aligned common map:", len(evidence_map), "rows")

In [ ]:
if RUN_FINAL_VALIDATION:
    for path, expected_hash in frozen_pilot_hashes.items():
        if file_sha256(path) != expected_hash:
            raise RuntimeError("STOP: frozen pilot file changed: " + path)
    validation_report = {
        "manifest_sha256": manifest_sha256, "ordering": "metadata_id ascending",
        "image": final_validations["image"], "text": final_validations["text"],
        "mapping_rows": len(evidence_map), "mapping_sha256": file_sha256(evidence_map_path),
        "pilot_artifacts_unchanged": True,
    }
    write_json_atomic(large_output_dir / "visualnews_train_final_validation.json", validation_report)
    print("Final validation passed. Pilot files unchanged.")

## Notes

- Set only the required stage flag to `True` and run its cells manually. All flags are saved as `False`.
- After a disconnect, rerun setup and Image Progress/Text Embeddings. Completed maps rebuild the skip set; unfinished images require rescanning the remote tar from its beginning.
- Checkpoints use arrays, aligned maps and `.complete.json` markers; completed files are never overwritten. A missing single file stops for repair.
- Failure counts count logged attempts. Failed images remain unfinished; do not silently drop them or merge fewer than 385,003 rows.
- Keep one writer per production directory. Never select `smoke_test/` for production or consolidation.
- The archive stays remote. No full image dataset is extracted or saved. Query/pilot artifacts remain read-only.
- This notebook prepares embeddings only. FAISS, retrieval, annotations and reranker training are later stages.